# AirWatch: Air Quality Analysis

**Important:** The bundled starter CSV is synthetic demo data. Replace it with the cited real dataset before reporting empirical findings.

## 1. Imports and load data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

INPUT = Path('../data/SYNTHETIC_demo_air_quality.csv')  # replace with real source CSV
raw = pd.read_csv(INPUT)
raw.head()

## 2. Inspect data quality

In [ ]:
print('Shape:', raw.shape)
print(raw.dtypes)
display(raw.isna().sum().rename('missing').to_frame())
print('Duplicate rows:', raw.duplicated().sum())

## 3. Clean and engineer features

In [ ]:
from clean_analyze import clean_data
clean = clean_data(raw)
clean.head()

## 4. Descriptive statistics

In [ ]:
display(clean['aqi'].describe().to_frame())
display(clean.groupby('city')['aqi'].agg(['count','mean','median','min','max']).round(2).sort_values('mean', ascending=False))

## 5. Visualizations

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=clean, x='city', y='aqi', errorbar=None)
plt.title('AQI observations by city (mean)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
monthly = clean.groupby(clean['date'].dt.to_period('M'))['aqi'].mean()
monthly.index = monthly.index.astype(str)
plt.figure(figsize=(11,5))
monthly.plot()
plt.title('Monthly mean AQI')
plt.ylabel('Mean AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
pivot = clean.pivot_table(index='city', columns='month_name', values='aqi', aggfunc='mean')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot = pivot.reindex(columns=[m for m in month_order if m in pivot.columns])
plt.figure(figsize=(12,5))
sns.heatmap(pivot, annot=False, cmap='YlOrRd')
plt.title('Mean AQI by city and month')
plt.tight_layout()

## 6. Export cleaned data for BI tools

In [ ]:
OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUT/'air_quality_cleaned.csv', index=False)
print('Saved:', OUT/'air_quality_cleaned.csv')

## 7. Interpretation checklist

- State the date range and cities actually present.
- Report measured results only after running on the real dataset.
- Discuss missingness and uneven station/city coverage.
- Distinguish association from causation.
- Do not claim a future prediction without validation.